# Day 3 RCA — Full AIOps Pipeline with AI-Powered RCA
## Observe → Think → Act
**AIOps Fundamentals Training | Microland**

---

## What This Capstone Demonstrates

This is the complete AIOps pipeline — from raw incident data to AI-generated
root cause analysis delivered automatically to an engineer's inbox.

```
OBSERVE                    THINK                        ACT
-------                    -----                        ---
Simulate a real         Elastic ML detects         Gemini AI performs
server incident    -->  the anomaly pattern  -->   Root Cause Analysis  -->  Email sent
Upload to Elastic       Query the incident         Generate RCA report       to engineer
                        timeline from ES            with recommendations
```

---

## The Incident Scenario

**Server:** `WINTEL-APP-07` — a Windows application server running a
customer-facing order management system.

**What happened (the full story):**

| Time | Event |
|------|-------|
| 02:15 | A scheduled database cleanup job starts and runs longer than expected |
| 02:30 | CPU begins climbing — cleanup job consuming excessive resources |
| 02:45 | Memory pressure builds as the cleanup job leaks handles |
| 03:00 | IIS application pool crashes due to memory exhaustion |
| 03:02 | Windows automatically restarts the application pool |
| 03:05 | Application pool crashes again — memory still exhausted |
| 03:08 | Service enters crash loop — users cannot access the order portal |
| 03:15 | First user complaint logged. Helpdesk raises INC0089541 |
| 03:45 | This notebook detects, analyses, and emails the RCA automatically |

---

## Tools Used in This Capstone

| Tool | Role | Cost |
|------|------|------|
| **Elastic Cloud** | Store and index all incident event data | Free trial |
| **Elastic ML** | Detect anomalies in the metric timeline | Included in trial |
| **Google AI Studio (Gemini)** | Perform AI-powered Root Cause Analysis | Free tier (60 req/min) |
| **Gmail SMTP / SendGrid** | Send the RCA report by email | Free |

---

## Before You Start — Get Your API Keys

### 1. Elastic Cloud
- Go to cloud.elastic.co → your deployment → copy Cloud ID and API Key

### 2. Google AI Studio (Gemini) — Free API Key
- Go to: **aistudio.google.com**
- Click **Get API Key** (top left)
- Click **Create API key** → copy the key
- Free tier: 60 requests per minute, 1,500 requests per day — more than enough


> **No Python knowledge required.** Run each cell in order using the Play button.


## Step 1 — Install Required Libraries


In [ ]:
!pip install elasticsearch google-generativeai --quiet
print('All libraries installed.')


## Step 2 — Configuration

Fill in all your credentials here. This is the only cell you need to edit.


In [ ]:
# ============================================================
# ELASTIC CLOUD
# ============================================================
CLOUD_ID   = 'Your Elastic Cloud ID'
API_KEY    = 'Your API Key'
INDEX_NAME = 'aiops-rca-using-gemini'

# ============================================================
# GOOGLE AI STUDIO (GEMINI)
# Get free key at: aistudio.google.com
# ============================================================
GEMINI_API_KEY = 'Your Gemini API key'
GEMINI_MODEL   = 'gemini-3.5-flash'   # Free, fast, excellent for analysis

# ============================================================
# INCIDENT DETAILS
# ============================================================
SERVER_NAME    = 'WINTEL-APP-07'
SERVICE_NAME   = 'Order Management Portal'
INCIDENT_ID    = 'INC0089541'

print('Configuration loaded.')
print(f'   Server    : {SERVER_NAME}')
print(f'   Service   : {SERVICE_NAME}')
print(f'   Incident  : {INCIDENT_ID}')
print(f'   AI Model  : {GEMINI_MODEL}')


## Step 3 — Connect to Elastic Cloud


In [ ]:
from elasticsearch import Elasticsearch

es = Elasticsearch(cloud_id=CLOUD_ID, api_key=API_KEY)
info = es.info()
print(f'Connected: {info["cluster_name"]} v{info["version"]["number"]}')


## Step 4 — Connect to Google AI Studio (Gemini)

This cell creates the connection to Gemini using your free API key.
Gemini will be used later to analyse the incident timeline and generate the RCA.


In [ ]:
import google.generativeai as genai



genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel(GEMINI_MODEL)

# Quick connectivity test
test_response = model.generate_content(
    'Reply with exactly this text and nothing else: Gemini connected successfully.'
)
print(f'Gemini API: {test_response.text.strip()}')
print(f'Model     : {GEMINI_MODEL}')


## Step 5 — OBSERVE: Create Index and Simulate the Incident

This is the **OBSERVE** phase of the AIOps pipeline.

We generate a realistic 6-hour timeline of events on WINTEL-APP-07:
- **Hours 0-2:** Normal baseline operation (the model learns this as 'normal')
- **Hour 2+:** The database cleanup job starts — CPU and memory begin climbing
- **Hour 3:** The crash loop begins — application pool crashes and restarts repeatedly
- **Hours 3-6:** Recovery attempts, sustained high resource usage, errors in logs

Three data streams are generated simultaneously — exactly as a real Metricbeat
and Winlogbeat deployment would produce:
1. **System metrics** — CPU, memory, disk I/O every 30 seconds
2. **Windows Event Logs** — Application errors, service crashes, restarts
3. **IIS/Application Logs** — HTTP errors, response times, failed requests


In [ ]:
import random
import math
from datetime import datetime, timedelta, timezone

random.seed(2025)

# Delete and recreate index
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

mapping = {
    'mappings': {
        'properties': {
            '@timestamp':              {'type': 'date'},
            'host.name':               {'type': 'keyword'},
            'event.type':              {'type': 'keyword'},
            'event.category':          {'type': 'keyword'},
            'event.dataset':           {'type': 'keyword'},
            'event.code':              {'type': 'keyword'},
            'event.outcome':           {'type': 'keyword'},
            'log.level':               {'type': 'keyword'},
            'message':                 {'type': 'text'},
            'system.cpu.total.pct':    {'type': 'float'},
            'system.memory.used.pct':  {'type': 'float'},
            'system.memory.swap.used.pct': {'type': 'float'},
            'system.diskio.read.bytes':{'type': 'long'},
            'process.name':            {'type': 'keyword'},
            'http.response.status_code': {'type': 'integer'},
            'http.response.time_ms':   {'type': 'float'},
            'error.type':              {'type': 'keyword'},
            'error.message':           {'type': 'text'},
            'incident.phase':          {'type': 'keyword'},
            'incident.id':             {'type': 'keyword'},
        }
    }
}
es.indices.create(index=INDEX_NAME, body=mapping)
print(f'Index created: {INDEX_NAME}')

# --- Timeline setup ---
now        = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
start_time = now - timedelta(hours=6)

# Key incident timestamps
T_CLEANUP_START  = start_time + timedelta(hours=2, minutes=15)
T_CPU_SPIKE      = start_time + timedelta(hours=2, minutes=30)
T_MEM_PRESSURE   = start_time + timedelta(hours=2, minutes=45)
T_FIRST_CRASH    = start_time + timedelta(hours=3, minutes=0)
T_CRASH_LOOP     = start_time + timedelta(hours=3, minutes=5)
T_SERVICE_DOWN   = start_time + timedelta(hours=3, minutes=8)
T_INCIDENT_RAISED = start_time + timedelta(hours=3, minutes=15)

all_events = []

# ===================================================================
# STREAM 1: System metrics every 30 seconds
# ===================================================================
total_metric_samples = int(6 * 3600 / 30)

for i in range(total_metric_samples):
    ts = start_time + timedelta(seconds=i * 30)
    elapsed_min = (ts - start_time).total_seconds() / 60

    if ts < T_CLEANUP_START:
        # Normal baseline
        cpu  = min(0.95, max(0.05, random.gauss(0.28, 0.04)))
        mem  = min(0.95, max(0.30, random.gauss(0.52, 0.03)))
        swap = max(0.0,  random.gauss(0.02, 0.01))
        phase = 'baseline'

    elif ts < T_MEM_PRESSURE:
        # Cleanup job starts — CPU climbs
        mins_in = (ts - T_CLEANUP_START).total_seconds() / 60
        cpu  = min(0.97, 0.28 + (mins_in / 15) * 0.55 + random.gauss(0, 0.03))
        mem  = min(0.97, 0.52 + (mins_in / 15) * 0.15 + random.gauss(0, 0.02))
        swap = max(0.0,  random.gauss(0.04, 0.01))
        phase = 'cpu_spike'

    elif ts < T_FIRST_CRASH:
        # Memory pressure builds
        mins_in = (ts - T_MEM_PRESSURE).total_seconds() / 60
        cpu  = min(0.97, 0.83 + random.gauss(0, 0.04))
        mem  = min(0.97, 0.67 + (mins_in / 15) * 0.28 + random.gauss(0, 0.02))
        swap = min(0.60, 0.05 + (mins_in / 15) * 0.30 + random.gauss(0, 0.02))
        phase = 'memory_pressure'

    elif ts < T_SERVICE_DOWN:
        # Crash loop — erratic metrics
        cpu  = min(0.99, max(0.30, random.gauss(0.75, 0.18)))
        mem  = min(0.99, max(0.40, random.gauss(0.88, 0.08)))
        swap = min(0.80, max(0.10, random.gauss(0.45, 0.10)))
        phase = 'crash_loop'

    else:
        # Service down / recovery attempts
        mins_in = (ts - T_SERVICE_DOWN).total_seconds() / 60
        cpu  = min(0.95, max(0.10, random.gauss(0.65, 0.15)))
        mem  = min(0.95, max(0.55, 0.92 - (mins_in / 180) * 0.30 + random.gauss(0, 0.04)))
        swap = min(0.70, max(0.05, random.gauss(0.35, 0.08)))
        phase = 'service_down'

    all_events.append({
        '@timestamp':              ts.isoformat(),
        'host.name':               SERVER_NAME,
        'event.type':              'metric',
        'event.category':          'host',
        'event.dataset':           'system.cpu',
        'log.level':               'info',
        'system.cpu.total.pct':    round(cpu, 4),
        'system.memory.used.pct':  round(mem, 4),
        'system.memory.swap.used.pct': round(swap, 4),
        'system.diskio.read.bytes': random.randint(1_000_000, 50_000_000),
        'incident.phase':          phase,
        'incident.id':             INCIDENT_ID,
        'message':                 f'System metrics: CPU={cpu*100:.1f}% MEM={mem*100:.1f}%'
    })

# ===================================================================
# STREAM 2: Windows Event Log entries
# ===================================================================
windows_events = [
    # Normal events (background)
    (start_time + timedelta(minutes=10),  '7036', 'information', 'normal',
     'The Windows Update service entered the running state.', 'services.exe'),
    (start_time + timedelta(minutes=45),  '4624', 'information', 'normal',
     'An account was successfully logged on. Account: svc_dbcleanup', 'lsass.exe'),
    (start_time + timedelta(minutes=90),  '4688', 'information', 'normal',
     'A new process has been created: sqlservr.exe', 'sqlservr.exe'),
    (start_time + timedelta(hours=1, minutes=30), '4624', 'information', 'normal',
     'An account was successfully logged on. Account: admin_it', 'lsass.exe'),

    # Cleanup job starts
    (T_CLEANUP_START, '4688', 'information', 'cpu_spike',
     'A new process has been created: db_cleanup_job.exe — scheduled task initiated',
     'db_cleanup_job.exe'),
    (T_CLEANUP_START + timedelta(minutes=5), '7036', 'warning', 'cpu_spike',
     'The SQL Server (MSSQLSERVER) service CPU usage exceeded 80%', 'sqlservr.exe'),

    # Memory pressure
    (T_MEM_PRESSURE, '2004', 'warning', 'memory_pressure',
     'Windows successfully diagnosed a low virtual memory condition. The following programs consumed the most virtual memory: db_cleanup_job.exe (1823 MB)',
     'db_cleanup_job.exe'),
    (T_MEM_PRESSURE + timedelta(minutes=5), '2019', 'error', 'memory_pressure',
     'The server was unable to allocate from the system nonpaged pool because the pool was empty.',
     'System'),

    # First crash
    (T_FIRST_CRASH, '1000', 'error', 'crash_loop',
     'Faulting application name: w3wp.exe (IIS Application Pool: OrderManagement), '
     'Faulting module name: ntdll.dll, Exception code: 0xc0000017 (STATUS_NO_MEMORY)',
     'w3wp.exe'),
    (T_FIRST_CRASH + timedelta(seconds=30), '7031', 'error', 'crash_loop',
     'The World Wide Web Publishing Service service terminated unexpectedly. '
     'It has done this 1 time(s).',
     'services.exe'),
    (T_FIRST_CRASH + timedelta(seconds=45), '7036', 'information', 'crash_loop',
     'The World Wide Web Publishing Service service entered the running state. (Auto-restart)',
     'services.exe'),

    # Crash loop
    (T_CRASH_LOOP, '1000', 'error', 'crash_loop',
     'Faulting application name: w3wp.exe — second crash. '
     'Memory not released from previous instance. Exception: 0xc0000017',
     'w3wp.exe'),
    (T_CRASH_LOOP + timedelta(seconds=20), '7031', 'error', 'crash_loop',
     'The World Wide Web Publishing Service service terminated unexpectedly. '
     'It has done this 2 time(s). The following corrective action will be taken in 0 ms.',
     'services.exe'),
    (T_CRASH_LOOP + timedelta(seconds=30), '7036', 'information', 'crash_loop',
     'The World Wide Web Publishing Service service entered the running state. (Auto-restart 2)',
     'services.exe'),

    # Service down
    (T_SERVICE_DOWN, '7034', 'error', 'service_down',
     'The World Wide Web Publishing Service service terminated unexpectedly. '
     'It has done this 3 time(s). No more automatic restarts will be attempted.',
     'services.exe'),
    (T_SERVICE_DOWN + timedelta(seconds=10), '1026', 'error', 'service_down',
     'Application: w3wp.exe — Framework: .NET Runtime Error — '
     'OutOfMemoryException: Failed to allocate managed memory for OrderManagement pool.',
     'w3wp.exe'),
    (T_SERVICE_DOWN + timedelta(minutes=2), '41', 'critical', 'service_down',
     'The system has rebooted without cleanly shutting down first. '
     'This error could be caused by the system running out of memory.',
     'Kernel-Power'),

    # Incident raised
    (T_INCIDENT_RAISED, '4624', 'information', 'service_down',
     f'L1 Engineer logged on to investigate. Incident {INCIDENT_ID} raised.',
     'lsass.exe'),
]

for ts, code_ev, level, phase, msg, proc in windows_events:
    all_events.append({
        '@timestamp':      ts.isoformat(),
        'host.name':       SERVER_NAME,
        'event.type':      'log',
        'event.category':  'process' if code_ev in ['4688','7036','7031','7034'] else 'authentication',
        'event.dataset':   'system.security' if code_ev.startswith('4') else 'system.application',
        'event.code':      code_ev,
        'log.level':       level,
        'process.name':    proc,
        'message':         msg,
        'incident.phase':  phase,
        'incident.id':     INCIDENT_ID,
        'system.cpu.total.pct':   None,
        'system.memory.used.pct': None,
    })

# ===================================================================
# STREAM 3: IIS / Application HTTP logs
# ===================================================================
http_events = []

# Normal HTTP traffic (baseline)
for i in range(30):
    ts = start_time + timedelta(minutes=random.randint(5, 130))
    http_events.append((ts, 200, random.uniform(120, 850), 'normal', '/api/orders'))

# Degraded responses during CPU spike
for i in range(20):
    ts = T_CPU_SPIKE + timedelta(minutes=random.randint(0, 15))
    http_events.append((ts, 200, random.uniform(2000, 8000), 'cpu_spike', '/api/orders'))
    http_events.append((ts + timedelta(seconds=30), 504, random.uniform(30000, 30000), 'cpu_spike', '/api/orders'))

# 500 errors during crash loop
for i in range(40):
    ts = T_FIRST_CRASH + timedelta(seconds=random.randint(0, 480))
    http_events.append((ts, 503, 0, 'crash_loop', '/api/orders'))
    http_events.append((ts, 500, 0, 'crash_loop', '/api/orderhistory'))

# Complete outage
for i in range(60):
    ts = T_SERVICE_DOWN + timedelta(minutes=random.randint(0, 120))
    http_events.append((ts, 503, 0, 'service_down', '/api/orders'))

for ts, status, resp_time, phase, endpoint in http_events:
    level = 'error' if status >= 500 else ('warning' if status >= 400 else 'info')
    all_events.append({
        '@timestamp':               ts.isoformat(),
        'host.name':                SERVER_NAME,
        'event.type':               'access',
        'event.category':           'web',
        'event.dataset':            'iis.access',
        'event.outcome':            'failure' if status >= 400 else 'success',
        'log.level':                level,
        'http.response.status_code': status,
        'http.response.time_ms':    round(resp_time, 1),
        'process.name':             'w3wp.exe',
        'message':                  f'HTTP {status} {endpoint} {resp_time:.0f}ms',
        'incident.phase':           phase,
        'incident.id':              INCIDENT_ID,
        'error.type':               'HTTP_503_ServiceUnavailable' if status == 503 else
                                    'HTTP_504_GatewayTimeout' if status == 504 else
                                    'HTTP_500_InternalError' if status == 500 else None,
    })

# Sort and upload
all_events.sort(key=lambda x: x['@timestamp'])
# Remove None values
clean_events = [{k: v for k, v in e.items() if v is not None} for e in all_events]

print(f'Generated {len(clean_events):,} incident events across 3 data streams:')
metrics = sum(1 for e in clean_events if e.get('event.type') == 'metric')
logs    = sum(1 for e in clean_events if e.get('event.type') == 'log')
http    = sum(1 for e in clean_events if e.get('event.type') == 'access')
print(f'   System metrics (Metricbeat)  : {metrics:,}')
print(f'   Windows Event Logs           : {logs:,}')
print(f'   IIS / Application HTTP logs  : {http:,}')
print()
print('Incident timeline:')
print(f'   {T_CLEANUP_START.strftime("%H:%M")} — DB cleanup job started')
print(f'   {T_CPU_SPIKE.strftime("%H:%M")} — CPU spike begins')
print(f'   {T_MEM_PRESSURE.strftime("%H:%M")} — Memory pressure builds')
print(f'   {T_FIRST_CRASH.strftime("%H:%M")} — First IIS crash')
print(f'   {T_CRASH_LOOP.strftime("%H:%M")} — Crash loop begins')
print(f'   {T_SERVICE_DOWN.strftime("%H:%M")} — Service fully down')
print(f'   {T_INCIDENT_RAISED.strftime("%H:%M")} — Incident {INCIDENT_ID} raised')


## Step 6 — Upload All Events to Elastic Cloud


In [ ]:
from elasticsearch.helpers import bulk

def generate_actions(docs, index):
    for doc in docs:
        yield {'_index': index, '_source': doc}

print(f'Uploading {len(clean_events):,} events to "{INDEX_NAME}" ...')
success, errors = bulk(es, generate_actions(clean_events, INDEX_NAME),
                       chunk_size=300, raise_on_error=False)
es.indices.refresh(index=INDEX_NAME)

print(f'Upload complete: {success:,} documents indexed, {len(errors)} errors')
print()
print('Data is now searchable in Kibana.')
print('Open Kibana > Discover and filter on index: aiops-capstone-rca')
print('You can see the full incident timeline across all three data streams.')


Uploading 948 events to "aiops-capstone-rca-version2" ...
Upload complete: 948 documents indexed, 0 errors

Data is now searchable in Kibana.
Open Kibana > Discover and filter on index: aiops-capstone-rca
You can see the full incident timeline across all three data streams.


## Step 7 — THINK: Query Elastic for the Incident Evidence

This is the **THINK** phase. We query Elasticsearch to extract:
1. The metric timeline — CPU and memory readings across the 6-hour window
2. All Windows Event Log errors and warnings
3. HTTP error counts by phase
4. A chronological sequence of key events

This evidence package is what we hand to Gemini for Root Cause Analysis.


In [ ]:
from datetime import timezone

print('Querying Elastic for incident evidence...')
print()

# --- 1. Peak CPU and Memory per phase ---
phase_stats = es.search(index=INDEX_NAME, body={
    'size': 0,
    'query': {'term': {'event.type': 'metric'}},
    'aggs': {
        'by_phase': {
            'terms': {'field': 'incident.phase', 'size': 10},
            'aggs': {
                'avg_cpu': {'avg': {'field': 'system.cpu.total.pct'}},
                'max_cpu': {'max': {'field': 'system.cpu.total.pct'}},
                'avg_mem': {'avg': {'field': 'system.memory.used.pct'}},
                'max_mem': {'max': {'field': 'system.memory.used.pct'}},
            }
        }
    }
})

# --- 2. Windows Event Log errors ---
error_events = es.search(index=INDEX_NAME, body={
    'size': 20,
    'query': {
        'bool': {
            'must': [{'term': {'event.type': 'log'}}],
            'should': [
                {'term': {'log.level': 'error'}},
                {'term': {'log.level': 'critical'}},
                {'term': {'log.level': 'warning'}}
            ],
            'minimum_should_match': 1
        }
    },
    'sort': [{'@timestamp': 'asc'}]
})

# --- 3. HTTP error counts ---
http_errors = es.search(index=INDEX_NAME, body={
    'size': 0,
    'query': {'term': {'event.type': 'access'}},
    'aggs': {
        'by_status': {
            'terms': {'field': 'http.response.status_code', 'size': 10}
        },
        'by_phase': {
            'terms': {'field': 'incident.phase', 'size': 10},
            'aggs': {
                'error_count': {
                    'filter': {'range': {'http.response.status_code': {'gte': 400}}}
                }
            }
        }
    }
})

# --- 4. First and last event timestamps ---
timeline_bounds = es.search(index=INDEX_NAME, body={
    'size': 0,
    'aggs': {
        'first_event': {'min': {'field': '@timestamp'}},
        'last_event':  {'max': {'field': '@timestamp'}}
    }
})

# Build evidence summary
phase_order = ['baseline', 'cpu_spike', 'memory_pressure', 'crash_loop', 'service_down']
phase_data = {}
for b in phase_stats['aggregations']['by_phase']['buckets']:
    phase_data[b['key']] = {
        'avg_cpu': round(b['avg_cpu']['value'] * 100, 1) if b['avg_cpu']['value'] else 0,
        'max_cpu': round(b['max_cpu']['value'] * 100, 1) if b['max_cpu']['value'] else 0,
        'avg_mem': round(b['avg_mem']['value'] * 100, 1) if b['avg_mem']['value'] else 0,
        'max_mem': round(b['max_mem']['value'] * 100, 1) if b['max_mem']['value'] else 0,
    }

error_log_entries = []
for hit in error_events['hits']['hits']:
    src = hit['_source']
    error_log_entries.append({
        'timestamp': src['@timestamp'][:19].replace('T', ' '),
        'level':     src.get('log.level', ''),
        'code':      src.get('event.code', ''),
        'process':   src.get('process.name', ''),
        'message':   src.get('message', '')[:200]
    })

http_status_counts = {}
for b in http_errors['aggregations']['by_status']['buckets']:
    http_status_counts[b['key']] = b['doc_count']

print('Evidence extracted from Elasticsearch:')
print()
print('Phase-by-phase metric summary:')
print(f'{"Phase":<20} {"Avg CPU":>9} {"Peak CPU":>9} {"Avg MEM":>9} {"Peak MEM":>9}')
print('-' * 60)
for phase in phase_order:
    if phase in phase_data:
        d = phase_data[phase]
        print(f'{phase:<20} {d["avg_cpu"]:>8.1f}% {d["max_cpu"]:>8.1f}% '
              f'{d["avg_mem"]:>8.1f}% {d["max_mem"]:>8.1f}%')
print()
print(f'Windows Event Log errors/warnings : {len(error_log_entries)}')
print(f'HTTP error responses              : {sum(v for k,v in http_status_counts.items() if k >= 400)}')
print(f'HTTP 503 (Service Unavailable)    : {http_status_counts.get(503, 0)}')
print(f'HTTP 504 (Gateway Timeout)        : {http_status_counts.get(504, 0)}')
print()
print('Key Windows Event Log entries:')
for entry in error_log_entries[:5]:
    print(f'  [{entry["timestamp"]}] {entry["level"].upper():8s} '
          f'Code:{entry["code"]:6s} {entry["message"][:100]}')


## Step 8 — THINK: Gemini AI Performs Root Cause Analysis

This is the **THINK** phase — the AI reasoning step.

We build a structured prompt containing all the evidence extracted from Elasticsearch
and send it to **Gemini 1.5 Flash** via the Google AI Studio API.

Gemini will analyse the evidence and produce:
- A plain-English summary of what happened
- The identified root cause
- A confidence level
- The contributing factors
- Immediate actions for the L1/L2 engineer
- Preventive recommendations to avoid recurrence


In [ ]:
print('Building incident evidence package for Gemini...')

# Build the evidence string
phase_summary_text = '\n'.join([
    f'  - {phase}: Avg CPU={phase_data.get(phase, {}).get("avg_cpu", "N/A")}%, '
    f'Peak CPU={phase_data.get(phase, {}).get("max_cpu", "N/A")}%, '
    f'Avg Memory={phase_data.get(phase, {}).get("avg_mem", "N/A")}%, '
    f'Peak Memory={phase_data.get(phase, {}).get("max_mem", "N/A")}%'
    for phase in phase_order if phase in phase_data
])

event_log_text = '\n'.join([
    f'  [{e["timestamp"]}] {e["level"].upper():8s} EventCode:{e["code"]:6s} '
    f'Process:{e["process"]:25s} | {e["message"][:180]}'
    for e in error_log_entries
])

http_text = '\n'.join([
    f'  HTTP {status}: {count} occurrences'
    for status, count in sorted(http_status_counts.items())
])

timeline_text = f"""
  {T_CLEANUP_START.strftime('%H:%M')} — Scheduled database cleanup job (db_cleanup_job.exe) started
  {T_CPU_SPIKE.strftime('%H:%M')}    — CPU began climbing rapidly (cleanup job consuming excessive resources)
  {T_MEM_PRESSURE.strftime('%H:%M')} — Memory pressure detected; virtual memory warning logged
  {T_FIRST_CRASH.strftime('%H:%M')}  — IIS w3wp.exe crashed (OutOfMemoryException, STATUS_NO_MEMORY)
  {T_CRASH_LOOP.strftime('%H:%M')}   — Second crash; service entered crash loop (Windows auto-restart)
  {T_SERVICE_DOWN.strftime('%H:%M')} — IIS W3SVC stopped after 3 crashes; no more automatic restarts
  {T_INCIDENT_RAISED.strftime('%H:%M')} — Incident {INCIDENT_ID} raised; L1 engineer logged on
"""

# Build the Gemini prompt
prompt = f"""You are an expert AIOps Root Cause Analysis (RCA) assistant.
Analyse the following IT incident evidence from an Elasticsearch AIOps platform
and produce a structured RCA report.

=== INCIDENT DETAILS ===
Incident ID   : {INCIDENT_ID}
Server        : {SERVER_NAME}
Service       : {SERVICE_NAME} (IIS-hosted .NET application)
OS            : Windows Server 2019
Incident Time : {T_SERVICE_DOWN.strftime('%Y-%m-%d %H:%M UTC')}
Current Status: Service unavailable — crash loop, IIS stopped

=== INCIDENT TIMELINE (from Elasticsearch) ===
{timeline_text}

=== SYSTEM METRICS BY PHASE (from Elastic ML anomaly detection) ===
{phase_summary_text}

=== WINDOWS EVENT LOG — ERRORS AND WARNINGS ===
{event_log_text}

=== HTTP RESPONSE CODE DISTRIBUTION ===
{http_text}

=== YOUR TASK ===
Produce a structured RCA report using EXACTLY this format:

## INCIDENT SUMMARY
(2-3 sentences describing what happened in plain English suitable for an L1 engineer)

## ROOT CAUSE
(One clear statement identifying the primary root cause)
Confidence: [HIGH / MEDIUM / LOW]
Evidence: (2-3 specific data points from the evidence above that confirm this root cause)

## CONTRIBUTING FACTORS
(Bullet list of 3-4 factors that made the situation worse)

## IMPACT ASSESSMENT
- Service affected: {SERVICE_NAME}
- Duration of impact: (calculate from the timeline above)
- Users affected: (estimate based on HTTP error counts)
- Severity: (P1/P2/P3 with justification)

## IMMEDIATE ACTIONS (for the L1/L2 Engineer Right Now)
(Numbered list of exactly 4 specific steps to restore service immediately)

## PREVENTIVE RECOMMENDATIONS (for the Change Management Team)
(Numbered list of exactly 4 specific changes to prevent recurrence)

## LESSONS LEARNED
(2-3 sentences on what this incident teaches about monitoring and automation gaps)

Keep the tone professional and technical but understandable to an L1 support engineer.
Be specific — reference actual process names, event codes, and metric values from the evidence.
"""

print(f'Sending {len(prompt):,} character evidence package to Gemini...')
print()

# Call Gemini
response = model.generate_content(prompt)
rca_report = response.text

print('Gemini RCA Report generated successfully.')
print('=' * 70)
print(rca_report)
print('=' * 70)


## Step 9 — Store the RCA Report Back in Elasticsearch

Best practice: store the AI-generated RCA alongside the incident data in Elasticsearch.
This creates an audit trail and allows future ML models to learn from past RCAs.


In [ ]:
from datetime import datetime, timezone

rca_doc = {
    '@timestamp':        datetime.now(timezone.utc).isoformat(),
    'host.name':         SERVER_NAME,
    'incident.id':       INCIDENT_ID,
    'incident.phase':    'rca_complete',
    'event.type':        'rca_report',
    'event.dataset':     'aiops.rca',
    'log.level':         'info',
    'message':           f'AI RCA completed for {INCIDENT_ID}',
    'rca.report':        rca_report,
    'rca.ai_model':      GEMINI_MODEL,
    'rca.generated_by':  'Google AI Studio — Gemini',
    'rca.evidence_docs': len(clean_events),
    'rca.prompt_chars':  len(prompt),
    'rca.report_chars':  len(rca_report),
}

es.index(index=INDEX_NAME, document=rca_doc)
es.indices.refresh(index=INDEX_NAME)

print('RCA report stored in Elasticsearch.')
print(f'   Index    : {INDEX_NAME}')
print(f'   Doc type : rca_report')
print(f'   Incident : {INCIDENT_ID}')
print()
total_docs = es.count(index=INDEX_NAME)['count']
print(f'Total documents in {INDEX_NAME}: {total_docs:,}')
print('  (includes metrics, event logs, HTTP logs, and the RCA report)')


## Step 11 — Visualise the Full Incident Timeline

This chart shows the complete incident as the Elastic ML model saw it —
CPU and memory over the full 6-hour window with the key incident markers.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime

# Extract metric data from uploaded events
metric_events = [e for e in clean_events if e.get('event.type') == 'metric']
timestamps  = [datetime.fromisoformat(e['@timestamp'].replace('Z', '+00:00'))
               for e in metric_events]
cpu_values  = [e.get('system.cpu.total.pct', 0) * 100 for e in metric_events]
mem_values  = [e.get('system.memory.used.pct', 0) * 100 for e in metric_events]
phases      = [e.get('incident.phase', '') for e in metric_events]

phase_colors = {
    'baseline':         '#1B3A6B',
    'cpu_spike':        '#F0A500',
    'memory_pressure':  '#C55A11',
    'crash_loop':       '#AA0000',
    'service_down':     '#660000',
}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True)

# Plot CPU
for phase, color in phase_colors.items():
    mask = [p == phase for p in phases]
    ts_p   = [t for t, m in zip(timestamps, mask) if m]
    cpu_p  = [c for c, m in zip(cpu_values, mask) if m]
    if ts_p:
        ax1.scatter(ts_p, cpu_p, c=color, s=3, alpha=0.7, zorder=2)

ax1.axhline(80, color='orange', linestyle='--', linewidth=1.2,
            alpha=0.7, label='80% warning threshold')
ax1.axhline(90, color='red', linestyle='--', linewidth=1.2,
            alpha=0.7, label='90% critical threshold')

# Annotate key events on CPU chart
for event_ts, label, y_pos in [
    (T_CLEANUP_START, 'Cleanup\nJob Start', 45),
    (T_FIRST_CRASH,   'First\nCrash',       50),
    (T_SERVICE_DOWN,  'Service\nDown',       55),
]:
    ax1.axvline(event_ts, color='#555555', linestyle=':', linewidth=1, alpha=0.6)
    ax1.annotate(label, xy=(event_ts, y_pos),
                xytext=(8, 0), textcoords='offset points',
                fontsize=7.5, color='#333333', fontweight='bold')

ax1.set_ylabel('CPU Utilisation %', fontsize=10)
ax1.set_title(
    f'AIOps Incident Timeline — {SERVER_NAME} | {INCIDENT_ID}\n'
    f'CPU and Memory across 6-hour window | Gemini AI RCA generated',
    fontsize=11, fontweight='bold', color='#1B3A6B'
)
ax1.set_ylim(0, 105)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))
ax1.grid(True, alpha=0.3)

legend_patches = [mpatches.Patch(color=c, label=p.replace('_', ' ').title())
                  for p, c in phase_colors.items()]
ax1.legend(handles=legend_patches, loc='upper left', fontsize=8, ncol=3)

# Plot Memory
for phase, color in phase_colors.items():
    mask = [p == phase for p in phases]
    ts_p  = [t for t, m in zip(timestamps, mask) if m]
    mem_p = [m for m, mk in zip(mem_values, mask) if mk]
    if ts_p:
        ax2.scatter(ts_p, mem_p, c=color, s=3, alpha=0.7, zorder=2)

ax2.axhline(85, color='orange', linestyle='--', linewidth=1.2, alpha=0.7)
ax2.axhline(95, color='red',    linestyle='--', linewidth=1.2, alpha=0.7)

for event_ts in [T_CLEANUP_START, T_FIRST_CRASH, T_SERVICE_DOWN]:
    ax2.axvline(event_ts, color='#555555', linestyle=':', linewidth=1, alpha=0.6)

ax2.set_ylabel('Memory Utilisation %', fontsize=10)
ax2.set_xlabel('Time (UTC)', fontsize=10)
ax2.set_ylim(0, 105)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))
ax2.grid(True, alpha=0.3)

import matplotlib.dates as mdates
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax2.xaxis.set_major_locator(mdates.MinuteLocator(byminute=[0, 30]))
fig.autofmt_xdate(rotation=30)
plt.tight_layout()
plt.show()

print('Timeline chart rendered.')
print('The colour phases show exactly what Elastic ML sees:')
for phase, color in phase_colors.items():
    print(f'  {phase.replace("_", " ").title():25s} = {color}')


## Step 12 — Capstone Summary: The Full AIOps Pipeline

Congratulations — you have just built and run a complete AIOps pipeline.

---

### What You Built

```
OBSERVE                         THINK                          ACT
=======                         =====                          ===

Simulated a realistic      Elasticsearch stored        Gemini AI analysed
server incident       -->  and indexed 750+       -->  the evidence and  --> Email sent
across 3 data streams       events. Queried            produced a full       to engineer
(metrics, event logs,       the incident               RCA report with       inbox
 HTTP logs)                 timeline and               root cause,
                            evidence                   actions, and
Uploaded to Elastic                                    prevention steps
Cloud (Elasticsearch)
```

---

### The Value Demonstrated

| Without AIOps | With This AIOps Pipeline |
|--------------|-------------------------|
| L1 engineer receives: 'Service down' | L1 engineer receives: Full RCA email with root cause, timeline, and 4 immediate actions |
| Engineer spends 30-60 min investigating | Investigation already done by AI in seconds |
| Root cause found after multiple escalations | Root cause (db_cleanup_job.exe memory leak) identified automatically |
| RCA report written manually after resolution | RCA report auto-generated and emailed before resolution |
| Next occurrence: same investigation repeats | Prevention recommendations stored for change management |

---

### Discussion Questions

**Q1:** The Gemini AI identified the root cause from log evidence.
As an L1 engineer, what would you do BEFORE implementing any of its recommendations?

**Q2:** The pipeline sent an RCA email automatically — without any human reviewing
the AI's analysis first. What governance process should be in place before this
goes into production?

**Q3:** What additional data sources would make the Gemini analysis more accurate?
(Think about what was missing from the evidence we provided)

**Q4:** The Observe-Think-Act pipeline you built took about 10 minutes to run.
In a real production environment, how quickly would you want this to complete
from the moment the incident starts?

**Q5:** Looking at the RCA email you received — would this be useful in your
current role? What would you change about the format or content?
